<a href="https://colab.research.google.com/github/HarithaGottumukkala/DATA266-1598_hw5/blob/main/DATA266_HW5_1598.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 — Fine-Tuning LLM using LoRA (PEFT)

## Assignment Objective

In this assignment, I will fine-tune a pretrained language model on the DialogSum dataset using LoRA (Low-Rank Adaptation). I will first test the original model to create a baseline, fine-tune it using LoRA, and then compare the summaries produced before and after training.

I will also compare two LoRA ranks, `r = 4` and `r = 16`, to understand how the adapter size affects the number of trainable parameters and the quality of the generated summaries.

### Personal Parameters

- **SID4:** 1598
- **SEED:** 1598
- **SLICE:** 598
- **HP_ID:** 2
- **CLS_A:** 8
- **CLS_B:** 5

## 0. Environment Setup

I will use Google Colab with a GPU runtime for this assignment. The required libraries include Hugging Face Transformers, Datasets, PEFT, and supporting packages for training and evaluation.

In [ ]:
%pip install -q transformers datasets peft accelerate sentencepiece evaluate rouge_score

In [ ]:
import importlib.util

torchao_available = importlib.util.find_spec("torchao") is not None

print("torchao available:", torchao_available)

torchao available: False


### 0.1 Reproducibility

To make the experiment reproducible, I use my assigned `SEED = 1598` for Python, NumPy, and PyTorch. Using the same seed helps keep random operations as consistent as possible across repeated runs.

In [ ]:
import os
import random
import numpy as np
import torch

# ---------------------------------------------------------
# Personal parameters
# ---------------------------------------------------------
SID4 = 1598
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ask PyTorch to use deterministic behavior where possible.
torch.use_deterministic_algorithms(True, warn_only=True)

print("Personal Parameters")
print("-" * 35)
print(f"SID4   : {SID4}")
print(f"SEED   : {SEED}")
print(f"SLICE  : {SLICE}")
print(f"HP_ID  : {HP_ID}")
print(f"CLS_A  : {CLS_A}")
print(f"CLS_B  : {CLS_B}")

Personal Parameters
-----------------------------------
SID4   : 1598
SEED   : 1598
SLICE  : 598
HP_ID  : 2
CLS_A  : 8
CLS_B  : 5


### 0.2 Runtime Verification

Before loading the dataset or model, I verify that the notebook is using a GPU. This is useful because LoRA fine-tuning is much faster on a GPU than on a CPU.

In [ ]:
print("Runtime Information")
print("-" * 50)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version     : {torch.version.cuda}")

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU memory       : {gpu_memory:.2f} GB")
else:
    print("GPU was not detected. Please change the Colab runtime to T4 GPU.")

Runtime Information
--------------------------------------------------
PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU              : Tesla T4
CUDA version     : 12.8
GPU memory       : 14.56 GB


In [ ]:
import transformers
import datasets
import peft
import accelerate

print("Library Versions")
print("-" * 50)
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Accelerate   : {accelerate.__version__}")
print(f"PyTorch      : {torch.__version__}")

Library Versions
--------------------------------------------------
Transformers : 5.16.1
Datasets     : 4.8.5
PEFT         : 0.20.0
Accelerate   : 1.14.0
PyTorch      : 2.11.0+cu128


## 1. Data Preparation

### Question 1

Load the DialogSum dataset and prepare it for dialogue summarization.

- **Input:** dialogue
- **Target:** summary
- Display two processed samples.

### Approach

The DialogSum dataset contains conversations together with human-written summaries. I will use the `dialogue` as the model input and the corresponding `summary` as the target output.

Before preprocessing the data, I first load the dataset and inspect its splits and columns.

In [ ]:
from datasets import load_dataset

# Load the DialogSum dataset provided in the assignment
dataset = load_dataset("neil-code/dialogsum-test")

print("DialogSum Dataset")
print("-" * 50)
print(dataset)

DialogSum Dataset
--------------------------------------------------
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


In [ ]:
print("Dataset Information")
print("-" * 50)

for split_name in dataset.keys():
    print(f"{split_name.capitalize():10s}: {len(dataset[split_name]):,} examples")

print("\nColumns:")
print(dataset["train"].column_names)

Dataset Information
--------------------------------------------------
Train     : 1,999 examples
Validation: 499 examples
Test      : 499 examples

Columns:
['id', 'dialogue', 'summary', 'topic']


### 1.1 Inspecting a Raw Example

Before preprocessing, I inspect one example to understand the original structure of the dataset.

In [ ]:
sample = dataset["train"][0]

print("Raw Training Example")
print("=" * 70)

print("\nID:")
print(sample["id"])

print("\nDIALOGUE:")
print(sample["dialogue"])

print("\nSUMMARY:")
print(sample["summary"])

print("\nTOPIC:")
print(sample["topic"])

Raw Training Example

ID:
train_0

DIALOGUE:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

SUMMARY:
Mr. Smith's g

### 1.2 Preparing Input and Target Text

For each example:

- The dialogue is used as the **input**.
- The human-written summary is used as the **target**.

I add a short summarization instruction before each dialogue so the pretrained FLAN-T5 model clearly understands the task.

In [ ]:
def prepare_example(example):
    """
    Convert one DialogSum example into the input-target format
    used for dialogue summarization.
    """

    input_text = (
        "Summarize the following dialogue:\n\n"
        + example["dialogue"]
    )

    target_text = example["summary"]

    return {
        "input_text": input_text,
        "target_text": target_text
    }


processed_dataset = dataset.map(prepare_example)

print("Preprocessing completed.")
print("\nNew columns:")
print(processed_dataset["train"].column_names)

Preprocessing completed.

New columns:
['id', 'dialogue', 'summary', 'topic', 'input_text', 'target_text']


### 1.3 Two Processed Samples

The following examples show the final input-target format that will later be used for fine-tuning.

In [ ]:
for i in range(2):
    example = processed_dataset["train"][i]

    print("=" * 80)
    print(f"PROCESSED SAMPLE {i + 1}")
    print("=" * 80)

    print("\nINPUT:")
    print(example["input_text"])

    print("\nTARGET:")
    print(example["target_text"])

    print()

PROCESSED SAMPLE 1

INPUT:
Summarize the following dialogue:

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

TARGE

## 2. Baseline Inference

### Question 2

Load a pretrained language model and generate summaries for two DialogSum dialogues before fine-tuning.

These outputs will be saved as the baseline so that they can later be compared with the LoRA fine-tuned model.

### Approach

I use `google/flan-t5-small` as the pretrained sequence-to-sequence model.

To make the comparison fair, I select two examples from the **test split**. These same two dialogues will be reused after fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

print(f"Loading tokenizer and model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = base_model.to(device)

print("\nModel loaded successfully.")
print(f"Device: {device}")

Loading tokenizer and model: google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Model loaded successfully.
Device: cuda


### 2.1 Selecting Evaluation Dialogues

I select two examples from the DialogSum **test split** using my assigned seed. These examples are kept fixed throughout the notebook so that the baseline and fine-tuned models are evaluated on exactly the same dialogues.

In [ ]:
# Select two reproducible test examples
rng = random.Random(SEED)

eval_indices = rng.sample(
    range(len(processed_dataset["test"])),
    2
)

print("Selected test indices:", eval_indices)

Selected test indices: [240, 216]


In [ ]:
eval_examples = [
    processed_dataset["test"][index]
    for index in eval_indices
]

print(f"Saved {len(eval_examples)} evaluation examples.")

Saved 2 evaluation examples.


### 2.2 Evaluation Examples

Before generating the baseline summaries, I display the two test dialogues and their reference summaries.

In [ ]:
for i, example in enumerate(eval_examples, start=1):
    print("=" * 80)
    print(f"EVALUATION EXAMPLE {i}")
    print("=" * 80)

    print("\nDIALOGUE:")
    print(example["dialogue"])

    print("\nREFERENCE SUMMARY:")
    print(example["summary"])

    print()

EVALUATION EXAMPLE 1

DIALOGUE:
#Person1#: Hello. Is this ABC Rent-a-car Company?
#Person2#: Yes, speaking. May I help you?
#Person1#: This morning we rented a car and we are on the way to Niagara Falls. I'm afraid we have a car accident near the border.
#Person2#: That's too bad. What kind of accident is it? Are you all right?
#Person1#: I'm all right, but my friend is seriously injured. Will you call an ambulance and the police?
#Person2#: OK. I'll do it right away, but tell me how it happened.
#Person1#: I ran into the guardrail when I turned to the left.

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

EVALUATION EXAMPLE 2

DIALOGUE:
#Person1#: We need to do a group report tomorrow. I need to go to the stationer to buy something for the posters.
#Person2#: You need to make posters?
#Person1#: absolutely. Every time when we do the reports, we love making some beautiful p

### 2.3 Generate Baseline Summaries

The pretrained model is now used without any additional training.

I use deterministic generation settings so that the same input produces a consistent output and the later comparison remains fair.

In [ ]:
def generate_summary(model, input_text):
    """
    Generate one summary using fixed decoding settings.
    """

    model.eval()

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4,
            do_sample=False
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
baseline_results = []

for i, example in enumerate(eval_examples, start=1):

    baseline_summary = generate_summary(
        base_model,
        example["input_text"]
    )

    baseline_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "baseline_summary": baseline_summary
    })

print("Baseline inference completed.")

Baseline inference completed.


### 2.4 Baseline Results

The following outputs show how the original pretrained FLAN-T5 model summarizes the two test dialogues before LoRA fine-tuning.

In [ ]:
for result in baseline_results:

    print("=" * 80)
    print(f"BASELINE EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nBASELINE MODEL SUMMARY:")
    print(result["baseline_summary"])

    print()

BASELINE EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

BASELINE MODEL SUMMARY:
I ran into the guardrail and got into a car.

BASELINE EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

BASELINE MODEL SUMMARY:
I need to do a group report tomorrow. I need to do a group report tomorrow. I need to go to the stationer to buy something for the posters. I need markers, color pencils, a pair of scissors, a pair of highlighter, rulers. I need markers, color pencils and highlighters. I need markers, color pencils and highlighters



In [ ]:
import json

with open("baseline_outputs.json", "w") as f:
    json.dump(
        baseline_results,
        f,
        indent=2
    )

print("Baseline outputs saved to baseline_outputs.json")

Baseline outputs saved to baseline_outputs.json


## 3. Apply LoRA

### Question 3

Apply LoRA (PEFT) to the pretrained model.

For the first experiment, I use:

- **LoRA rank (`r`)**: 4
- **LoRA alpha**: 16
- **LoRA dropout**: 0.05
- **Target modules**: query (`q`) and value (`v`) projection layers

After attaching the LoRA adapters, I report the total number of parameters and the number of trainable parameters.

### Approach

LoRA keeps the original pretrained model weights frozen and adds small trainable adapter matrices to selected layers.

This allows the model to learn the dialogue summarization task while updating only a small fraction of the full model parameters.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

print("PEFT components imported successfully.")

PEFT components imported successfully.


### 3.1 LoRA Configuration

For the first experiment, I use `r = 4`. The adapters are attached to the query and value projection layers of the attention mechanism.

The original FLAN-T5 weights remain frozen while the LoRA adapter parameters are trainable.

In [ ]:
# LoRA configuration for Experiment 1
LORA_RANK = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

lora_config_r4 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none"
)

print("LoRA Configuration")
print("-" * 50)
print(f"Rank (r)        : {LORA_RANK}")
print(f"Alpha           : {LORA_ALPHA}")
print(f"Dropout         : {LORA_DROPOUT}")
print('Target modules  : ["q", "v"]')
print("Task type       : SEQ_2_SEQ_LM")

LoRA Configuration
--------------------------------------------------
Rank (r)        : 4
Alpha           : 16
Dropout         : 0.05
Target modules  : ["q", "v"]
Task type       : SEQ_2_SEQ_LM


### 3.2 Load a Fresh Base Model

I load a fresh copy of FLAN-T5-small before attaching LoRA. This keeps the baseline model separate and ensures that each LoRA rank experiment starts from the same pretrained model.

In [ ]:
# Load a fresh pretrained model for the r=4 LoRA experiment
model_r4 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model_r4 = model_r4.to(device)

print("Fresh FLAN-T5-small model loaded for the r=4 experiment.")
print(f"Device: {device}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh FLAN-T5-small model loaded for the r=4 experiment.
Device: cuda


In [ ]:
import importlib.util

print("torchao installed:", importlib.util.find_spec("torchao") is not None)

torchao installed: False


In [ ]:
model_r4 = get_peft_model(
    model_r4,
    lora_config_r4
)

print("LoRA adapters attached successfully.")

LoRA adapters attached successfully.


### 3.3 Trainable Parameters

To verify that LoRA is parameter-efficient, I compare the total number of model parameters with the number of parameters that will actually be updated during training.

Only the LoRA adapter parameters should be trainable, while the original FLAN-T5 parameters remain frozen.

In [ ]:
def count_parameters(model):
    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    trainable_percentage = (
        100 * trainable_params / total_params
    )

    return total_params, trainable_params, trainable_percentage


total_params_r4, trainable_params_r4, trainable_percent_r4 = count_parameters(
    model_r4
)

print("Parameter Summary — LoRA r=4")
print("-" * 50)
print(f"Total parameters     : {total_params_r4:,}")
print(f"Trainable parameters : {trainable_params_r4:,}")
print(f"Trainable percentage : {trainable_percent_r4:.4f}%")

Parameter Summary — LoRA r=4
--------------------------------------------------
Total parameters     : 77,133,184
Trainable parameters : 172,032
Trainable percentage : 0.2230%


### 3.4 Verifying the LoRA Parameters

I also inspect a few trainable parameter names to confirm that the parameters being updated belong to the LoRA adapters.

In [ ]:
print("First 10 Trainable Parameters")
print("-" * 70)

count = 0

for name, parameter in model_r4.named_parameters():
    if parameter.requires_grad:
        print(name)
        count += 1

    if count == 10:
        break

First 10 Trainable Parameters
----------------------------------------------------------------------
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_B.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_A.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_B.default.weight
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_B.default.weight


In [ ]:
experiment_r4_info = {
    "model": MODEL_NAME,
    "rank": LORA_RANK,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": ["q", "v"],
    "total_parameters": total_params_r4,
    "trainable_parameters": trainable_params_r4,
    "trainable_percentage": trainable_percent_r4
}

print("r=4 experiment information saved.")

r=4 experiment information saved.


### Observation

The number of trainable parameters is much smaller than the total number of model parameters. This confirms that LoRA is updating only a small adapter portion of the model instead of fine-tuning all FLAN-T5 parameters.

## 4. Fine-Tuning

### Question 4

Fine-tune the LoRA-enabled FLAN-T5 model on the DialogSum dataset.

A smaller subset is used to keep the experiment practical in Google Colab. The same training data and settings will later be reused for the `r = 16` experiment to make the comparison fair.

After training, I report the training loss and save the LoRA adapter.

### 4.1 Preparing the Training Subset

I use my assigned seed when selecting the training and validation examples. This keeps the subset reproducible across repeated runs.

The same subset will be used for both LoRA rank experiments.

In [ ]:
TRAIN_SIZE = 2000
VAL_SIZE = 300

# Shuffle reproducibly using the assigned seed
shuffled_train = processed_dataset["train"].shuffle(seed=SEED)
shuffled_validation = processed_dataset["validation"].shuffle(seed=SEED)

# Use a manageable subset for Colab training
train_subset = shuffled_train.select(
    range(min(TRAIN_SIZE, len(shuffled_train)))
)

validation_subset = shuffled_validation.select(
    range(min(VAL_SIZE, len(shuffled_validation)))
)

print("Training Subset")
print("-" * 45)
print(f"Training examples   : {len(train_subset):,}")
print(f"Validation examples : {len(validation_subset):,}")
print(f"Seed                : {SEED}")

Training Subset
---------------------------------------------
Training examples   : 1,999
Validation examples : 300
Seed                : 1598


### 4.2 Tokenizing the Data

The model cannot directly process raw text. Therefore, I convert the input dialogues and target summaries into token IDs.

The dialogue input is limited to 512 tokens, while the target summary is limited to 128 tokens.

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


def tokenize_batch(batch):
    """
    Tokenize the dialogue inputs and reference summaries.
    """

    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train = train_subset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_subset.column_names
)

tokenized_validation = validation_subset.map(
    tokenize_batch,
    batched=True,
    remove_columns=validation_subset.column_names
)

print("Tokenization completed.")
print()
print("Training columns:")
print(tokenized_train.column_names)

print("\nValidation columns:")
print(tokenized_validation.column_names)

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenization completed.

Training columns:
['input_ids', 'attention_mask', 'labels']

Validation columns:
['input_ids', 'attention_mask', 'labels']


### 4.3 Checking the Tokenized Data

Before training, I inspect one tokenized example to confirm that the model receives input IDs, an attention mask, and target labels.

In [ ]:
sample_tokenized = tokenized_train[0]

print("Tokenized Example")
print("-" * 50)
print(f"Input token count : {len(sample_tokenized['input_ids'])}")
print(f"Target token count: {len(sample_tokenized['labels'])}")

print("\nAvailable fields:")
print(sample_tokenized.keys())

Tokenized Example
--------------------------------------------------
Input token count : 347
Target token count: 68

Available fields:
dict_keys(['input_ids', 'attention_mask', 'labels'])


### 4.4 Dynamic Padding

A data collator is used to dynamically pad examples within each training batch. This avoids padding every example to the maximum possible length and uses memory more efficiently.

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r4
)

print("Data collator created successfully.")

Data collator created successfully.


### 4.5 Training Configuration

I train the LoRA adapter for two epochs using the same reproducible settings that will later be used for the `r = 16` experiment.

Only the LoRA parameters are updated during training.

In [ ]:
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR_R4 = "dialogsum_lora_r4"

training_args_r4 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_R4,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Training arguments created.")

Training arguments created.


In [ ]:
from transformers import Seq2SeqTrainer

trainer_r4 = Seq2SeqTrainer(
    model=model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer created successfully for LoRA r=4.")

Trainer created successfully for LoRA r=4.


### 4.6 Fine-Tuning the `r = 4` Adapter

The model is now trained on the selected DialogSum examples.

During training, the original FLAN-T5 parameters stay frozen and only the LoRA adapter parameters are updated.

In [ ]:
train_result_r4 = trainer_r4.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan


In [ ]:
final_training_loss_r4 = train_result_r4.training_loss

print("LoRA r=4 Training Complete")
print("-" * 50)
print(f"Final training loss : {final_training_loss_r4:.4f}")
print(f"Global training steps: {train_result_r4.global_step}")

LoRA r=4 Training Complete
--------------------------------------------------
Final training loss : 0.0000
Global training steps: 250


### 4.6 Training Sanity Check

The first training attempt produced a zero training loss and `NaN` validation loss, which indicates that the run was not numerically stable.

Before retraining, I verify that the target labels were created correctly.

In [ ]:
example = tokenized_train[0]

print("Sanity Check")
print("-" * 50)

print(f"Input tokens : {len(example['input_ids'])}")
print(f"Target tokens: {len(example['labels'])}")

print("\nDecoded target:")
print(
    tokenizer.decode(
        example["labels"],
        skip_special_tokens=True
    )
)

Sanity Check
--------------------------------------------------
Input tokens : 347
Target tokens: 68

Decoded target:
#Person1# interviews Susan Hill, a Londoner who lives and works in Italy, on #Person1#'s program, Working Abroad. Susan went to Italy in 1982 and got a job there. She now works as a designer. She thinks her industry has become more competitive since 1982.


### 4.7 Retraining with Stable Precision

The first run showed numerical instability when using FP16. I therefore restart the `r = 4` experiment from the original pretrained FLAN-T5 weights and train using FP32.

All other main experimental settings remain unchanged.

In [ ]:
import gc

del trainer_r4
del model_r4

gc.collect()
torch.cuda.empty_cache()

print("Previous unstable training run cleared from memory.")

Previous unstable training run cleared from memory.


In [ ]:
model_r4 = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model_r4 = get_peft_model(
    model_r4,
    lora_config_r4
)

model_r4 = model_r4.to(device)

print("Fresh LoRA r=4 model created.")
model_r4.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh LoRA r=4 model created.
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


In [ ]:
data_collator_r4 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r4
)

print("Data collator recreated.")

Data collator recreated.


In [ ]:
training_args_r4 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_R4,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=False,

    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Stable FP32 training configuration created.")

Stable FP32 training configuration created.


In [ ]:
trainer_r4 = Seq2SeqTrainer(
    model=model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator_r4,
    processing_class=tokenizer
)

print("Trainer recreated successfully.")

Trainer recreated successfully.


In [ ]:
test_batch = data_collator_r4(
    [tokenized_train[0], tokenized_train[1]]
)

test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

model_r4.eval()

with torch.no_grad():
    test_output = model_r4(**test_batch)

print("Pre-training batch loss:")
print(test_output.loss.item())

Pre-training batch loss:
2.3330459594726562


### 4.8 Final `r = 4` Training Run

The FP32 sanity check produced a valid positive loss, so I now run the complete fine-tuning experiment.

Before training, I remove the checkpoints from the failed FP16 attempt so that the final results contain only the stable FP32 run.

In [ ]:
import os
import shutil

# Remove checkpoints created by the earlier unstable FP16 run
if os.path.exists(OUTPUT_DIR_R4):
    shutil.rmtree(OUTPUT_DIR_R4)

print("Old r=4 training checkpoints cleared.")

Old r=4 training checkpoints cleared.


In [ ]:
import time

start_time_r4 = time.time()

train_result_r4 = trainer_r4.train()

training_time_r4 = time.time() - start_time_r4

print("\nTraining finished.")
print(f"Training time: {training_time_r4 / 60:.2f} minutes")

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch,Training Loss,Validation Loss
1,3.271652,1.430061
2,3.228856,1.405656



Training finished.
Training time: 2.47 minutes


In [ ]:
final_training_loss_r4 = train_result_r4.training_loss

print("LoRA r=4 Training Summary")
print("-" * 50)
print(f"Final training loss  : {final_training_loss_r4:.4f}")
print(f"Global training steps: {train_result_r4.global_step}")
print(f"Training time        : {training_time_r4 / 60:.2f} minutes")

LoRA r=4 Training Summary
--------------------------------------------------
Final training loss  : 3.4798
Global training steps: 250
Training time        : 2.47 minutes


In [ ]:
eval_results_r4 = trainer_r4.evaluate()

final_eval_loss_r4 = eval_results_r4["eval_loss"]

print("LoRA r=4 Evaluation")
print("-" * 50)
print(f"Validation loss: {final_eval_loss_r4:.4f}")

Training Loss,Validation Loss,Epoch
3.228856,1.405656,2


LoRA r=4 Evaluation
--------------------------------------------------
Validation loss: 1.4057


In [ ]:
R4_ADAPTER_PATH = "dialogsum_lora_r4_adapter"

model_r4.save_pretrained(R4_ADAPTER_PATH)
tokenizer.save_pretrained(R4_ADAPTER_PATH)

print(f"LoRA r=4 adapter saved to: {R4_ADAPTER_PATH}")

LoRA r=4 adapter saved to: dialogsum_lora_r4_adapter


In [ ]:
experiment_r4_info.update({
    "training_examples": len(train_subset),
    "validation_examples": len(validation_subset),
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "precision": "FP32",
    "final_training_loss": final_training_loss_r4,
    "final_validation_loss": final_eval_loss_r4,
    "global_steps": train_result_r4.global_step,
    "training_time_minutes": training_time_r4 / 60
})

print("Final r=4 metrics saved.")

Final r=4 metrics saved.


## 5. Post-Fine-Tuning Inference

### Question 5

Run inference using the fine-tuned LoRA model on the same two dialogues used for the baseline experiment.

Using the same examples and generation settings allows me to directly compare the pretrained model with the fine-tuned model.

### Approach

I reuse the two test dialogues selected earlier. I also keep the same decoding settings used during baseline inference so that the only major difference is the fine-tuned LoRA adapter.

In [ ]:
model_r4.eval()

print("Fine-tuned LoRA r=4 model is ready for inference.")

Fine-tuned LoRA r=4 model is ready for inference.


In [ ]:
r4_results = []

for i, example in enumerate(eval_examples, start=1):

    fine_tuned_summary = generate_summary(
        model_r4,
        example["input_text"]
    )

    r4_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "fine_tuned_summary": fine_tuned_summary
    })

print("Post-fine-tuning inference completed.")

Post-fine-tuning inference completed.


### 5.1 Fine-Tuned Model Outputs

The following outputs are generated by the `r = 4` LoRA fine-tuned model using the same two test dialogues used for the baseline.

In [ ]:
for result in r4_results:

    print("=" * 80)
    print(f"FINE-TUNED EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nFINE-TUNED r=4 SUMMARY:")
    print(result["fine_tuned_summary"])

    print()

FINE-TUNED EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

FINE-TUNED r=4 SUMMARY:
#Person1# rented a car and they are on the way to Niagara Falls. They have a car accident near the border. #Person2# will call an ambulance and the police.

FINE-TUNED EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

FINE-TUNED r=4 SUMMARY:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# needs markers, color pencils and highlighters and pencils. #Person2# needs markers, color pencils and highlighters and pencils.



In [ ]:
with open("r4_finetuned_outputs.json", "w") as f:
    json.dump(
        r4_results,
        f,
        indent=2
    )

print("Fine-tuned r=4 outputs saved to r4_finetuned_outputs.json")

Fine-tuned r=4 outputs saved to r4_finetuned_outputs.json


In [ ]:
print("Evaluation Examples Used")
print("-" * 40)

for i, index in enumerate(eval_indices, start=1):
    print(f"Example {i}: test index {index}")

Evaluation Examples Used
----------------------------------------
Example 1: test index 240
Example 2: test index 216


## 6. Output Comparison

### Question 6

Compare the summaries generated before LoRA fine-tuning with the summaries generated after fine-tuning.

The same two test dialogues are used so that the change in model behavior can be compared fairly.

In [ ]:
import pandas as pd

comparison_rows = []

for i in range(2):
    comparison_rows.append({
        "Example": i + 1,
        "Test Index": eval_indices[i],
        "Reference Summary": baseline_results[i]["reference_summary"],
        "Baseline Summary": baseline_results[i]["baseline_summary"],
        "LoRA r=4 Summary": r4_results[i]["fine_tuned_summary"]
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,Example,Test Index,Reference Summary,Baseline Summary,LoRA r=4 Summary
0,1,240,#Person1# rent a car from ABC Rent-a-car Compa...,I ran into the guardrail and got into a car.,#Person1# rented a car and they are on the way...
1,2,216,#Person1# and #Person2# will do a group report...,I need to do a group report tomorrow. I need t...,#Person1# and #Person2# need to do a group rep...


In [ ]:
for i in range(2):
    print("=" * 90)
    print(f"EXAMPLE {i + 1} — TEST INDEX {eval_indices[i]}")
    print("=" * 90)

    print("\nREFERENCE:")
    print(baseline_results[i]["reference_summary"])

    print("\nBEFORE FINE-TUNING:")
    print(baseline_results[i]["baseline_summary"])

    print("\nAFTER FINE-TUNING — LoRA r=4:")
    print(r4_results[i]["fine_tuned_summary"])

    print()

EXAMPLE 1 — TEST INDEX 240

REFERENCE:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

BEFORE FINE-TUNING:
I ran into the guardrail and got into a car.

AFTER FINE-TUNING — LoRA r=4:
#Person1# rented a car and they are on the way to Niagara Falls. They have a car accident near the border. #Person2# will call an ambulance and the police.

EXAMPLE 2 — TEST INDEX 216

REFERENCE:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

BEFORE FINE-TUNING:
I need to do a group report tomorrow. I need to do a group report tomorrow. I need to go to the stationer to buy something for the posters. I need markers, color pencils, a pair of scissors, a pair of highlighter, rulers. I need markers, color pencils and highlighters. I need markers, color pencils and highlighters

AFTER

### Observation

After LoRA fine-tuning, the first summary became more informative and captured important details such as the car accident and the call for an ambulance and police. The second example also captured the main group-report context, but it repeated some shopping items and missed a few details from the reference summary. Overall, fine-tuning improved task-specific summarization, although the model still showed some repetition and information loss.

In [ ]:
comparison_df.to_csv(
    "baseline_vs_r4_comparison.csv",
    index=False
)

print("Comparison saved to baseline_vs_r4_comparison.csv")

Comparison saved to baseline_vs_r4_comparison.csv


## 7. LoRA Rank Experiment

### Question 7

Compare two LoRA ranks, `r = 4` and `r = 16`.

The goal is to observe how increasing the LoRA rank changes the number of trainable parameters, training behavior, and summary quality.

To keep the experiment fair, the same model, dataset subset, training settings, and evaluation examples are used for both ranks.

### Approach

The `r = 4` experiment is already complete.

For the second experiment, I load a fresh FLAN-T5-small model and attach a new LoRA adapter with `r = 16`. All other main settings remain the same.

In [ ]:
import gc

del model_r4
del trainer_r4

gc.collect()
torch.cuda.empty_cache()

print("r=4 model cleared from GPU memory.")

r=4 model cleared from GPU memory.


In [ ]:
LORA_RANK_R16 = 16

lora_config_r16 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK_R16,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none"
)

print("LoRA r=16 Configuration")
print("-" * 50)
print(f"Rank (r)        : {LORA_RANK_R16}")
print(f"Alpha           : {LORA_ALPHA}")
print(f"Dropout         : {LORA_DROPOUT}")
print('Target modules  : ["q", "v"]')

LoRA r=16 Configuration
--------------------------------------------------
Rank (r)        : 16
Alpha           : 16
Dropout         : 0.05
Target modules  : ["q", "v"]


In [ ]:
model_r16 = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model_r16 = get_peft_model(
    model_r16,
    lora_config_r16
)

model_r16 = model_r16.to(device)

print("Fresh LoRA r=16 model created.")
model_r16.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh LoRA r=16 model created.
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [ ]:
total_params_r16, trainable_params_r16, trainable_percent_r16 = count_parameters(
    model_r16
)

print("Parameter Summary — LoRA r=16")
print("-" * 50)
print(f"Total parameters     : {total_params_r16:,}")
print(f"Trainable parameters : {trainable_params_r16:,}")
print(f"Trainable percentage : {trainable_percent_r16:.4f}%")

Parameter Summary — LoRA r=16
--------------------------------------------------
Total parameters     : 77,649,280
Trainable parameters : 688,128
Trainable percentage : 0.8862%


### 7.1 Parameter Difference

Increasing the LoRA rank gives the adapter more trainable capacity. This increases the number of trainable parameters while still keeping most of the original FLAN-T5 model frozen.

In [ ]:
data_collator_r16 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r16
)

print("Data collator created for r=16.")

Data collator created for r=16.


In [ ]:
OUTPUT_DIR_R16 = "dialogsum_lora_r16"

training_args_r16 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_R16,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=False,

    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Training arguments created for r=16.")

Training arguments created for r=16.


In [ ]:
trainer_r16 = Seq2SeqTrainer(
    model=model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator_r16,
    processing_class=tokenizer
)

print("Trainer created successfully for LoRA r=16.")

Trainer created successfully for LoRA r=16.


In [ ]:
test_batch_r16 = data_collator_r16(
    [tokenized_train[0], tokenized_train[1]]
)

test_batch_r16 = {
    key: value.to(device)
    for key, value in test_batch_r16.items()
}

model_r16.eval()

with torch.no_grad():
    test_output_r16 = model_r16(**test_batch_r16)

print("Pre-training r=16 batch loss:")
print(test_output_r16.loss.item())

Pre-training r=16 batch loss:
2.3330459594726562


### 7.2 Fine-Tuning `r = 16`

The `r = 16` model is trained using exactly the same DialogSum subset and training settings as the `r = 4` model.

In [ ]:
import time

start_time_r16 = time.time()

train_result_r16 = trainer_r16.train()

training_time_r16 = time.time() - start_time_r16

print("\nr=16 training finished.")
print(f"Training time: {training_time_r16 / 60:.2f} minutes")

Epoch,Training Loss,Validation Loss
1,3.288325,1.439063
2,3.238898,1.412309



r=16 training finished.
Training time: 2.66 minutes


In [ ]:
final_training_loss_r16 = train_result_r16.training_loss

eval_results_r16 = trainer_r16.evaluate()
final_eval_loss_r16 = eval_results_r16["eval_loss"]

print("LoRA r=16 Training Summary")
print("-" * 50)
print(f"Final training loss   : {final_training_loss_r16:.4f}")
print(f"Validation loss       : {final_eval_loss_r16:.4f}")
print(f"Global training steps : {train_result_r16.global_step}")
print(f"Training time         : {training_time_r16 / 60:.2f} minutes")

Training Loss,Validation Loss,Epoch
3.238898,1.412309,2


LoRA r=16 Training Summary
--------------------------------------------------
Final training loss   : 3.4900
Validation loss       : 1.4123
Global training steps : 250
Training time         : 2.66 minutes


In [ ]:
R16_ADAPTER_PATH = "dialogsum_lora_r16_adapter"

model_r16.save_pretrained(R16_ADAPTER_PATH)
tokenizer.save_pretrained(R16_ADAPTER_PATH)

print(f"LoRA r=16 adapter saved to: {R16_ADAPTER_PATH}")

LoRA r=16 adapter saved to: dialogsum_lora_r16_adapter


In [ ]:
experiment_r16_info = {
    "model": MODEL_NAME,
    "rank": 16,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": ["q", "v"],
    "total_parameters": total_params_r16,
    "trainable_parameters": trainable_params_r16,
    "trainable_percentage": trainable_percent_r16,
    "training_examples": len(train_subset),
    "validation_examples": len(validation_subset),
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "precision": "FP32",
    "final_training_loss": final_training_loss_r16,
    "final_validation_loss": final_eval_loss_r16,
    "global_steps": train_result_r16.global_step,
    "training_time_minutes": training_time_r16 / 60
}

print("r=16 experiment information saved.")

r=16 experiment information saved.


### 7.3 Inference with the `r = 16` Model

I evaluate the `r = 16` model using the same two test dialogues used for the baseline and `r = 4` experiments.

The generation settings are kept unchanged so that the output quality can be compared fairly.

In [ ]:
model_r16.eval()

r16_results = []

for i, example in enumerate(eval_examples, start=1):

    r16_summary = generate_summary(
        model_r16,
        example["input_text"]
    )

    r16_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "fine_tuned_summary": r16_summary
    })

print("Post-fine-tuning inference completed for LoRA r=16.")

Post-fine-tuning inference completed for LoRA r=16.


### 7.4 `r = 16` Generated Summaries

The following outputs show the summaries generated by the higher-rank LoRA adapter.

In [ ]:
for result in r16_results:

    print("=" * 80)
    print(f"LoRA r=16 — EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nLoRA r=16 SUMMARY:")
    print(result["fine_tuned_summary"])

    print()

LoRA r=16 — EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

LoRA r=16 SUMMARY:
#Person1# and #Person2# are on the way to Niagara Falls. They have a car accident near the border. #Person1# will call an ambulance and the police.

LoRA r=16 — EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

LoRA r=16 SUMMARY:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# needs markers, color pencils and highlighters and pencils. #Person2# needs markers, color pencils and highlighters and pencils.



In [ ]:
with open("r16_finetuned_outputs.json", "w") as f:
    json.dump(
        r16_results,
        f,
        indent=2
    )

print("Fine-tuned r=16 outputs saved to r16_finetuned_outputs.json")

Fine-tuned r=16 outputs saved to r16_finetuned_outputs.json


### 7.5 Comparing `r = 4` and `r = 16`

I first compare the two experiments using their parameter counts, training loss, validation loss, and training time.

In [ ]:
rank_metrics_df = pd.DataFrame([
    {
        "LoRA Rank": 4,
        "Trainable Parameters": experiment_r4_info["trainable_parameters"],
        "Trainable %": experiment_r4_info["trainable_percentage"],
        "Training Loss": experiment_r4_info["final_training_loss"],
        "Validation Loss": experiment_r4_info["final_validation_loss"],
        "Training Time (min)": experiment_r4_info["training_time_minutes"]
    },
    {
        "LoRA Rank": 16,
        "Trainable Parameters": experiment_r16_info["trainable_parameters"],
        "Trainable %": experiment_r16_info["trainable_percentage"],
        "Training Loss": experiment_r16_info["final_training_loss"],
        "Validation Loss": experiment_r16_info["final_validation_loss"],
        "Training Time (min)": experiment_r16_info["training_time_minutes"]
    }
])

display(rank_metrics_df)

,LoRA Rank,Trainable Parameters,Trainable %,Training Loss,Validation Loss,Training Time (min)
0,4,172032,0.223032,3.479817,1.405656,2.466716
1,16,688128,0.886200,3.490043,1.412309,2.660842


In [ ]:
rank_metrics_df.to_csv(
    "lora_rank_metrics_comparison.csv",
    index=False
)

print("Rank metrics saved to lora_rank_metrics_comparison.csv")

Rank metrics saved to lora_rank_metrics_comparison.csv


In [ ]:
rank_output_rows = []

for i in range(2):
    rank_output_rows.append({
        "Example": i + 1,
        "Reference": baseline_results[i]["reference_summary"],
        "LoRA r=4": r4_results[i]["fine_tuned_summary"],
        "LoRA r=16": r16_results[i]["fine_tuned_summary"]
    })

rank_output_df = pd.DataFrame(rank_output_rows)

display(rank_output_df)

,Example,Reference,LoRA r=4,LoRA r=16
0,1,#Person1# rent a car from ABC Rent-a-car Compa...,#Person1# rented a car and they are on the way...,#Person1# and #Person2# are on the way to Niag...
1,2,#Person1# and #Person2# will do a group report...,#Person1# and #Person2# need to do a group rep...,#Person1# and #Person2# need to do a group rep...


In [ ]:
for i in range(2):

    print("=" * 90)
    print(f"RANK COMPARISON — EXAMPLE {i + 1}")
    print("=" * 90)

    print("\nREFERENCE:")
    print(baseline_results[i]["reference_summary"])

    print("\nLoRA r=4:")
    print(r4_results[i]["fine_tuned_summary"])

    print("\nLoRA r=16:")
    print(r16_results[i]["fine_tuned_summary"])

    print()

RANK COMPARISON — EXAMPLE 1

REFERENCE:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

LoRA r=4:
#Person1# rented a car and they are on the way to Niagara Falls. They have a car accident near the border. #Person2# will call an ambulance and the police.

LoRA r=16:
#Person1# and #Person2# are on the way to Niagara Falls. They have a car accident near the border. #Person1# will call an ambulance and the police.

RANK COMPARISON — EXAMPLE 2

REFERENCE:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

LoRA r=4:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# needs markers, color pencils and highlighters and pencils. #Person2# needs markers, color pencils and highlighters and

In [ ]:
rank_output_df.to_csv(
    "r4_vs_r16_output_comparison.csv",
    index=False
)

print("r=4 vs r=16 output comparison saved.")

r=4 vs r=16 output comparison saved.
